# 00 · Fuentes y descarga

¿Qué datos usamos y qué tan completos son?

La fuente principal es el **Cierre de la Producción Agrícola** del SIAP (Agricultura): una fila por cultivo, estado (y municipio desde 2003), ciclo, modalidad de riego y unidad de medida. En 2026 el portal se mudó de `infosiap.siap.gob.mx`, que ya no resuelve, a `nube.agricultura.gob.mx`.

| Fuente | Cobertura | Nivel | Script |
|---|---|---|---|
| SIAP, Cierre agrícola estatal | 1980-2002 | Estado | `src.descarga --solo siap` |
| SIAP, Cierre agrícola municipal | 2003-2025 | Municipio | `src.descarga --solo siap` |
| Nominatim + OSRM | Municipios productores 2021-2025 | Ruta por carretera a Hermosillo | `src.descarga --solo rutas` |
| Prensa y literatura | 1963-2026 | Cifras puntuales | `datos/manual/cifras_documentales.csv` |

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import plotly.graph_objects as go

from src import config, tema

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

estatal = pd.read_csv(config.DIR_PROCESSED / "cempasuchil_estatal.csv")
municipal = pd.read_csv(config.DIR_PROCESSED / "cempasuchil_municipal.csv")
FUENTE_SIAP = "Fuente: SIAP, Cierre de la Producción Agrícola 1980-2025. Análisis: Sahuaris."

In [2]:
archivos = sorted((config.DIR_RAW / "siap").rglob("*.csv"))
inventario = pd.DataFrame({
    "archivo": [a.name for a in archivos],
    "nivel": [a.parent.name for a in archivos],
    "MB": [round(a.stat().st_size / 1e6, 1) for a in archivos],
})
print(f"{len(archivos)} archivos, {inventario['MB'].sum():.0f} MB")
inventario.groupby("nivel").agg(archivos=("archivo", "count"), MB=("MB", "sum"))

46 archivos, 146 MB


,archivos,MB
nivel,,
estatal,23,9.0
municipal,23,137.5


## Cómo se llama el cempasúchil en el SIAP

El SIAP lo registra como **Zempoalxochitl** y lo divide en varios "cultivos" según cómo se vende. Para comparar entre años lo agrupamos en cuatro presentaciones (`src/procesamiento.py`):

- **Flor a granel (t):** toneladas de flor. En los 80 y 90 era sobre todo materia prima para pigmento; desde 2000, flor de corte.
- **Forrajero (t):** flor sembrada explícitamente para alimento animal (pigmento de la yema y la piel del pollo).
- **Flor en manojo:** manojos y gruesas, la flor de corte que se vende en mercados y panteones.
- **Planta en maceta:** la que se vende en maceta; crece mucho en CDMX y el Estado de México.

Las unidades no se pueden sumar entre sí. La medida común es la **superficie sembrada (hectáreas)**.

In [3]:
(estatal.groupby(["nomcultivo", "nomunidad", "presentacion"])
    .agg(primer_anio=("anio", "min"), ultimo_anio=("anio", "max"), anios=("anio", "nunique"))
    .sort_values("primer_anio"))

primer_anio  ultimo_anio  anios
nomcultivo               nomunidad presentacion                                      
Zempoalxochitl           Tonelada  Flor a granel (t)         1980         2025     46
Zempoalxochitl forrajero Tonelada  Forrajero (t)             1986         2001     13
Zempoalxochitl           Manojo    Flor en manojo            1997         2020     22
Zempoalxochitl (manojo)  Manojo    Flor en manojo            2003         2025      7
Zempoalxochitl           Planta    Planta en maceta          2005         2019     15
                         Gruesa    Flor en manojo            2013         2019      7
Zempoalxochitl (gruesa)  Gruesa    Flor en manojo            2022         2023      2
Zempoalxochitl (planta)  Planta    Planta en maceta          2022         2025      4

## Años con registro y número de estados productores

In [4]:
cobertura = estatal.groupby("anio").agg(
    estados=("nomestado", "nunique"),
    hectareas=("sembrada", "sum"),
)
cobertura["municipios"] = municipal.groupby("anio")["idmunicipio"].size()
faltantes = sorted(set(range(1980, 2026)) - set(cobertura.index))
print("Años sin ningún registro de cempasúchil:", faltantes or "ninguno")
cobertura.T

Años sin ningún registro de cempasúchil: ninguno


anio,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
estados,8.0,9.0,9.0,7.0,7.0,8.0,8.0,9.0,9.0,10.0,6.0,7.0,10.0,9.0,10.0,...,10.0,13.00,13.0,12.0,12.00,14.0,14.00,14.00,13.00,13.00,13.00,14.00,15.00,13.00,11.00
hectareas,5444.0,7458.0,8415.0,4333.0,9848.0,17577.0,6200.0,9523.0,17506.0,10232.0,5254.0,4475.0,14957.0,15819.0,7564.0,...,1193.7,1917.85,1668.1,1352.6,1950.25,1852.8,1934.09,2423.02,2185.52,2147.01,2210.68,2289.38,2470.93,2392.13,1771.05
municipios,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,53.0,71.00,75.0,66.0,83.00,98.0,106.00,116.00,111.00,104.00,113.00,122.00,143.00,134.00,110.00


## Huecos conocidos

- **Plantas en maceta 2020-2021:** el SIAP no las registra en ninguno de los dos años, aunque sí antes y después. Es un hueco de la fuente, no una caída real.
- **Antes de 2003** solo hay nivel estatal: no sabemos qué municipios de Sonora producían en los 80.
- **Plantas en maceta:** el SIAP cuenta muchas menos que el gobierno de la CDMX (notebook 05).

In [5]:
plantas = estatal[estatal["presentacion"] == "Planta en maceta"].groupby("anio")["volumenproduccion"].sum()
assert 2020 not in plantas.index and 2021 not in plantas.index
plantas.astype(int)

anio
2005     360000
2006     720000
2007     918000
2008     711500
2009     718000
2010     840195
2011    1050100
2012     720000
2013    1735000
2014    1920444
2015    2100728
2016    1827544
2017    2656400
2018    2192062
2019    2308825
2022    2313816
2023    2368200
2024    3163428
2025    3021200
Name: volumenproduccion, dtype: int64